# Заполнение пропусков в данных о жилье

Сравниваем три способа заполнения пропусков и ищем необычные дома через IsolationForest.
Цель – `SalePrice`, цена продажи. Для Cart Scope это отдельный пример подготовки данных для регрессии.

## Содержание

1. [Данные](#data)
2. [Разделение выборки](#split)
3. [Конвейер](#pipeline)
4. [Сравнение импутации](#comparison)
5. [Выбросы](#outliers)
6. [Оценка на тесте](#evaluation)
7. [Сохранение модели](#saving)

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data/raw/housing_prices.csv"
DATA_PATH = ROOT / "data/processed/housing_prices_incomplete.csv"
MODEL_PATH = ROOT / "models/housing_regression.pkl"

<a id="data"></a>
## Данные

Используем House Prices: 1 460 домов из Ames, штат Айова, с известной ценой продажи.
[Источник CSV](https://github.com/INRIA/scikit-learn-mooc/blob/1b294d95296e7e52dec5b9c69076f8010b2bc51f/datasets/house_prices.csv) – репозиторий курса scikit-learn от Inria.

Создать пропуски: `python -m src.data.create_incomplete_housing`.
В `LotArea`, `GrLivArea` и `TotalBsmtSF` случайно скрыты по 292 значения – ровно 20%.
Выбор позиций не зависит от цены и признаков: схема MCAR, seed 42.
Строки и набор столбцов сохранены.

В источнике `?` означает отсутствующее значение. Категория `None` в `MasVnrType`
означает отсутствие облицовки и остаётся категорией.

In [ ]:
read_options = {"keep_default_na": False, "na_values": ["?", ""]}
raw = pd.read_csv(RAW_PATH, **read_options)
data = pd.read_csv(DATA_PATH, **read_options)
print("Размер:", data.shape)
print("Повторы без Id:", raw.drop(columns="Id").duplicated().sum())
missing_table = pd.DataFrame({
    "Исходные пропуски": raw.isna().sum(),
    "После скрытия": data.isna().sum(),
})
missing_table["Добавлено"] = missing_table["После скрытия"] - missing_table["Исходные пропуски"]
display(missing_table.loc[missing_table["После скрытия"] > 0])
display(data.head())

<a id="split"></a>
## Разделение выборки

Оставляем 20% домов для итоговой оценки. `Id` не используем как признак.
Все методы сравниваем на пяти одинаковых фолдах обучающей выборки.

In [ ]:
X = data.drop(columns=["Id", "SalePrice"])
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
categorical_columns = X_train.select_dtypes(include="object").columns.tolist() + ["MSSubClass", "MoSold"]
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]
cv = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(X_train))
print("Обучение:", len(X_train), "Тест:", len(X_test))
print("Числовых признаков:", len(numeric_columns), "Категориальных:", len(categorical_columns))

<a id="pipeline"></a>
## Конвейер

Числовые признаки масштабируем до заполнения: это важно для расстояний в KNN.
Категориальные пропуски заменяем меткой `Missing` и применяем one-hot.
Такая метка сохраняет информацию об отсутствующем или неописанном объекте, например гараже.
Числовые коды `MSSubClass` и `MoSold` обрабатываем как категории.

Во всех вариантах одинаковый RandomForestRegressor: 150 деревьев, минимум 2 объекта в листе.
Масштабирование, импутация и кодирование обучаются внутри каждого фолда.

In [ ]:
def make_pipeline(method):
    imputers = {
        "Median": SimpleImputer(strategy="median"),
        "KNN": KNNImputer(n_neighbors=5, weights="distance"),
        "Iterative": IterativeImputer(max_iter=100, tol=0.01, n_nearest_features=None, random_state=SEED),
    }
    numeric = Pipeline([
        ("scaler", StandardScaler()),
        ("imputer", imputers[method]),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric, numeric_columns),
        ("categorical", categorical, categorical_columns),
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=SEED, n_jobs=1)),
    ])

<a id="comparison"></a>
## Сравнение импутации

Главная метрика – RMSE в долларах. Чем меньше, тем лучше.
Показываем среднее и стандартное отклонение по пяти фолдам.

In [ ]:
rows = []
for method in ["Median", "KNN", "Iterative"]:
    scores = cross_validate(
        make_pipeline(method), X_train, y_train, cv=cv,
        scoring={"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"},
        n_jobs=1, error_score="raise",
    )
    rows.append({
        "Метод": method,
        "RMSE": -scores["test_rmse"].mean(),
        "Разброс RMSE": scores["test_rmse"].std(ddof=1),
        "MAE": -scores["test_mae"].mean(),
        "Секунд на fit": scores["fit_time"].mean(),
    })
    print(method, "готово", flush=True)
comparison = pd.DataFrame(rows).set_index("Метод")
display(comparison.round(2))
best_method = comparison["RMSE"].idxmin()
print("Лучший метод:", best_method)
final_model = make_pipeline(best_method).fit(X_train, y_train)

<a id="outliers"></a>
## Выбросы

IsolationForest обучаем на подготовленных признаках train, без цены продажи.
`contamination=0.03` выделяет около 3% необычных объектов; это не доказательство ошибки.
PCA служит только для отображения. Аномалии из данных не удаляем.

In [ ]:
encoded = final_model.named_steps["preprocessor"].transform(X_train)
detector = IsolationForest(n_estimators=200, contamination=0.03, random_state=SEED, n_jobs=1)
is_outlier = detector.fit_predict(encoded) == -1
pca = PCA(n_components=2, random_state=SEED)
coordinates = pca.fit_transform(encoded)
fig, ax = plt.subplots(figsize=(8, 5))
for flag, label, color in [(False, "Остальные", "steelblue"), (True, "Аномальные", "tomato")]:
    mask = is_outlier == flag
    ax.scatter(coordinates[mask, 0], coordinates[mask, 1], s=18, alpha=0.65, color=color, label=label)
ax.set(xlabel="PC1", ylabel="PC2", title="Аномалии в признаках домов")
ax.legend()
plt.tight_layout()
plt.show()
explained_variance = pca.explained_variance_ratio_.sum()
print("Найдено аномалий:", int(is_outlier.sum()), "из", len(X_train))
print(f"Две компоненты объясняют {explained_variance:.1%} дисперсии.")
display(X_train.loc[is_outlier, ["LotArea", "GrLivArea", "OverallQual", "YearBuilt"]].head(10))

<a id="evaluation"></a>
## Оценка на тесте

Метод выбран по кросс-валидации. На тесте оцениваем полный конвейер один раз,
сохраняя все дома, включая необычные.

In [ ]:
prediction = final_model.predict(X_test)
test_metrics = pd.Series({
    "RMSE": root_mean_squared_error(y_test, prediction),
    "MAE": mean_absolute_error(y_test, prediction),
    "R²": r2_score(y_test, prediction),
}, name="Тест")
display(test_metrics.round(3))
print("Оценено домов:", len(prediction), "из", len(X_test))

In [ ]:
display(Markdown(
    f"По среднему RMSE выбран **{best_method}**: "
    f"**{comparison.loc[best_method, 'RMSE']:,.0f}** на кросс-валидации. "
    f"На тесте RMSE **{test_metrics['RMSE']:,.0f}**, MAE **{test_metrics['MAE']:,.0f}** долларов. "
    f"IsolationForest выделил **{int(is_outlier.sum())}** необычных домов; "
    f"PCA показывает **{explained_variance:.1%}** дисперсии."
))

KNN дал наименьший средний RMSE: 30 749 против 31 076 у медианы и 30 992 у Iterative.
Разница меньше 330 долларов при разбросе по фолдам около 5 300 долларов.
Это не подтверждает устойчивого преимущества KNN на других разбиениях.

Медиана проста и быстра, но не использует связи между признаками.
KNN ищет похожие дома, Iterative восстанавливает значения по другим признакам.
Дополнительная сложность не гарантирует меньшую ошибку.

IsolationForest выделил 36 домов. На PCA часть из них лежит на краях облака,
часть смешана с остальными. Две компоненты показывают только 24,3% дисперсии,
поэтому график не отражает все различия. Найденные аномалии не удалялись.

Сравнение относится к одному разбиению и случайно созданным пропускам.
Оценка на домах из Ames не описывает качество для других городов или будущих продаж.

<a id="saving"></a>
## Сохранение модели

Сохраняем предобработку и RandomForestRegressor одним файлом `pickle`.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as stream:
    pickle.dump(final_model, stream, protocol=pickle.HIGHEST_PROTOCOL)
print("Модель сохранена:", MODEL_PATH.relative_to(ROOT))